# actor_critic

> Actor and Critic for PPO with AAC agent.

In [15]:
#| default_exp agent.aac.actor_critic

In [16]:
#| export
import torch
import torch.nn as nn
from torch.distributions import Normal

In [17]:
#| export
class ActorCritic(nn.Module):
    def __init__(self,  num_actor_obs,
                        num_critic_obs,
                        num_actions,
                        actor_hidden_dims=[256, 256, 256],
                        critic_hidden_dims=[256, 256, 256],
                        init_noise_std=1.0,
                        activation = nn.ELU(),
                        **kwargs):
        if kwargs:
            print("ActorCritic.__init__ got unexpected arguments, which will be ignored: " + str([key for key in kwargs.keys()]))
        super(ActorCritic, self).__init__()


        mlp_input_dim_a = num_actor_obs
        mlp_input_dim_c = num_critic_obs
        # Policy
        actor_layers = []
        actor_layers.append(nn.Linear(mlp_input_dim_a, actor_hidden_dims[0]))
        actor_layers.append(activation)
        for l in range(len(actor_hidden_dims)):
            if l == len(actor_hidden_dims) - 1:
                actor_layers.append(nn.Linear(actor_hidden_dims[l], num_actions))
            else:
                actor_layers.append(nn.Linear(actor_hidden_dims[l], actor_hidden_dims[l + 1]))
                actor_layers.append(activation)
        self.actor = nn.Sequential(*actor_layers)

        # Value function
        critic_layers = []
        critic_layers.append(nn.Linear(mlp_input_dim_c, critic_hidden_dims[0]))
        critic_layers.append(activation)
        for l in range(len(critic_hidden_dims)):
            if l == len(critic_hidden_dims) - 1:
                critic_layers.append(nn.Linear(critic_hidden_dims[l], 1))
            else:
                critic_layers.append(nn.Linear(critic_hidden_dims[l], critic_hidden_dims[l + 1]))
                critic_layers.append(activation)
        self.critic = nn.Sequential(*critic_layers)

        print(f"Actor MLP: {self.actor}")
        print(f"Critic MLP: {self.critic}")

        # Action noise
        self.std = nn.Parameter(init_noise_std * torch.ones(num_actions))
        self.distribution = None
        # disable args validation for speedup
        Normal.set_default_validate_args = False
        

    @staticmethod
    # not used at the moment
    def init_weights(sequential, scales):
        [torch.nn.init.orthogonal_(module.weight, gain=scales[idx]) for idx, module in
         enumerate(mod for mod in sequential if isinstance(mod, nn.Linear))]


    def reset(self, dones=None):
        pass

    def forward(self):
        raise NotImplementedError
    
    @property
    def action_mean(self):
        return self.distribution.mean

    @property
    def action_std(self):
        return self.distribution.stddev
    
    @property
    def entropy(self):
        return self.distribution.entropy().sum(dim=-1)

    def update_distribution(self, observations):
        mean = self.actor(observations)
        self.distribution = Normal(mean, mean*0. + self.std)

    def act(self, observations, **kwargs):
        self.update_distribution(observations)
        return self.distribution.sample()
    
    def get_actions_log_prob(self, actions):
        return self.distribution.log_prob(actions).sum(dim=-1)

    def act_inference(self, observations):
        actions_mean = self.actor(observations)
        return actions_mean

    def evaluate(self, critic_observations, **kwargs):
        value = self.critic(critic_observations)
        return value

In [18]:
#| hide
from nbdev.showdoc import show_doc

In [19]:
show_doc(ActorCritic.init_weights)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/actor_critic.py#L72){target="_blank" style="float:right; font-size:smaller"}

### ActorCritic.init_weights

>      ActorCritic.init_weights (sequential, scales)

|    | **Details** |
| -- | ----------- |
| sequential |  |
| scales | not used at the moment |

In [20]:
show_doc(ActorCritic.reset)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/actor_critic.py#L80){target="_blank" style="float:right; font-size:smaller"}

### ActorCritic.reset

>      ActorCritic.reset (dones=None)

In [21]:
show_doc(ActorCritic.forward)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/actor_critic.py#L83){target="_blank" style="float:right; font-size:smaller"}

### ActorCritic.forward

>      ActorCritic.forward ()

*Define the computation performed at every call.

Should be overridden by all subclasses.

.. note::
    Although the recipe for forward pass needs to be defined within
    this function, one should call the :class:`Module` instance afterwards
    instead of this since the former takes care of running the
    registered hooks while the latter silently ignores them.*

In [22]:
show_doc(ActorCritic.update_distribution)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/actor_critic.py#L98){target="_blank" style="float:right; font-size:smaller"}

### ActorCritic.update_distribution

>      ActorCritic.update_distribution (observations)

In [23]:
show_doc(ActorCritic.act)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/actor_critic.py#L102){target="_blank" style="float:right; font-size:smaller"}

### ActorCritic.act

>      ActorCritic.act (observations, **kwargs)

In [24]:
show_doc(ActorCritic.get_actions_log_prob)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/actor_critic.py#L106){target="_blank" style="float:right; font-size:smaller"}

### ActorCritic.get_actions_log_prob

>      ActorCritic.get_actions_log_prob (actions)

In [25]:
show_doc(ActorCritic.act_inference)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/actor_critic.py#L109){target="_blank" style="float:right; font-size:smaller"}

### ActorCritic.act_inference

>      ActorCritic.act_inference (observations)

In [26]:
show_doc(ActorCritic.evaluate)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/actor_critic.py#L113){target="_blank" style="float:right; font-size:smaller"}

### ActorCritic.evaluate

>      ActorCritic.evaluate (critic_observations, **kwargs)

In [27]:
#| hide
import nbdev; nbdev.nbdev_export()